In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Telecom_KPI_Analysis") \
    .config("spark.jars.packages", "com.crealytics:spark-excel_2.13:3.5.1_0.20.4") \
    .getOrCreate()
#Load Telecom KPI Dataset
df = spark.read \
    .format("com.crealytics.spark.excel") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(r"C:\Users\91838\Desktop\telecom project\telecom data for pyspark.xlsx")

df.show(5)

+-------------+-------+-------------------+------+------------------+--------------------------+-----------------+-----------------------+------------------+-------------------+----+------------------+-------------------+------------------+----------+---------------+
|      Cell ID|SITE ID|               Date|Sector|Cell Availabililty|Session Setup Success Rate|VoLTE_Drop_Rate_%|Handover_Success_Rate_%|Traffic-24Hrs [GB]|DL_PRB_utilization%| CQI|IP_Throughput_Mbps|RRC Connected Users|Peak RRC Connected|Average_TA|Mute_Call Rate%|
+-------------+-------+-------------------+------+------------------+--------------------------+-----------------+-----------------------+------------------+-------------------+----+------------------+-------------------+------------------+----------+---------------+
|Site_A_CELL_1| Site_A|2025-01-01 00:00:00|   0.0|             94.81|                     99.96|              0.0|                  96.82|             95.12|              85.95|9.61|              

In [3]:
#data cleansing
from pyspark.sql.functions import col, to_date
df_clean = df.dropna(subset=["Cell ID", "SITE ID"])
df_clean = df_clean.filter(
    (col("Cell Availabililty") >= 0) &   # spelling SAME as file
    (col("Session Setup Success Rate") <= 100)
)

df_clean = df_clean.withColumn(
    "Date",
    to_date(col("Date"), "d-MMM-yy")
)
df_clean.show(5)

+-------------+-------+----------+------+------------------+--------------------------+-----------------+-----------------------+------------------+-------------------+----+------------------+-------------------+------------------+----------+---------------+
|      Cell ID|SITE ID|      Date|Sector|Cell Availabililty|Session Setup Success Rate|VoLTE_Drop_Rate_%|Handover_Success_Rate_%|Traffic-24Hrs [GB]|DL_PRB_utilization%| CQI|IP_Throughput_Mbps|RRC Connected Users|Peak RRC Connected|Average_TA|Mute_Call Rate%|
+-------------+-------+----------+------+------------------+--------------------------+-----------------+-----------------------+------------------+-------------------+----+------------------+-------------------+------------------+----------+---------------+
|Site_A_CELL_1| Site_A|2025-01-01|   0.0|             94.81|                     99.96|              0.0|                  96.82|             95.12|              85.95|9.61|              4.32|              64.11|           

In [4]:
# -------- 4. Site level Aggregation ---------

from pyspark.sql.functions import avg, sum, max

site_daily = df_clean.groupBy("SITE ID","Date").agg(
avg("Cell Availabililty").alias("Avg_Cell_Availability"),
avg("Session Setup Success Rate").alias("Avg_Session_SR"),
avg("VoLTE_Drop_Rate_%").alias("Avg_VoLTE_Drop"),
avg("Handover_Success_Rate_%").alias("Avg_HO_SR"),
sum("Traffic-24Hrs [GB]").alias("Total_Traffic_GB"),
avg("DL_PRB_utilization%").alias("Avg_PRB_Utilization"),
avg("CQI").alias("Avg_CQI"),
avg("IP_Throughput_Mbps").alias("Avg_Throughput"),
max("Peak RRC Connected").alias("Peak_RRC")
)

site_daily.show()

+-------+----------+---------------------+-----------------+--------------------+-----------------+------------------+-------------------+-----------------+-----------------+--------+
|SITE ID|      Date|Avg_Cell_Availability|   Avg_Session_SR|      Avg_VoLTE_Drop|        Avg_HO_SR|  Total_Traffic_GB|Avg_PRB_Utilization|          Avg_CQI|   Avg_Throughput|Peak_RRC|
+-------+----------+---------------------+-----------------+--------------------+-----------------+------------------+-------------------+-----------------+-----------------+--------+
| Site_A|2025-01-01|    92.39461538461539|99.80692307692308|0.033846153846153845|96.66307692307691|1202.3400000000001|  71.75153846153846|9.594615384615384|7.648461538461539|   121.0|
| Site_B|2025-01-01|             86.59875|          99.7475|             0.04125|         93.03625| 587.6899999999999|            71.9425|            9.255|          6.63875|    98.0|
+-------+----------+---------------------+-----------------+--------------------

In [5]:
# -------- 5. Detecting Network Congestion ---------

congestion_cells = df_clean.filter(
    (col("DL_PRB_utilization%") > 85) &
    (col("IP_Throughput_Mbps") < 5)
    )

congestion_cells.select(
    "Cell ID","SITE ID","DL_PRB_utilization%","IP_Throughput_Mbps"
    ).show()


+--------------+-------+-------------------+------------------+
|       Cell ID|SITE ID|DL_PRB_utilization%|IP_Throughput_Mbps|
+--------------+-------+-------------------+------------------+
| Site_A_CELL_1| Site_A|              85.95|              4.32|
| Site_A_CELL_3| Site_A|              91.97|              3.16|
| Site_A_CELL_4| Site_A|              93.84|              2.87|
| Site_A_CELL_6| Site_A|              85.56|              3.27|
|Site_A_CELL_11| Site_A|               88.9|              2.25|
| Site_B_CELL_2| Site_B|              86.02|              3.78|
| Site_B_CELL_5| Site_B|              86.98|              4.22|
| Site_B_CELL_8| Site_B|              87.72|              2.59|
+--------------+-------+-------------------+------------------+



In [6]:
# --------  6. VoLTE Quality Issues ---------

volte_issue_cells = df_clean.filter(
    (col("VoLTE_Drop_Rate_%") > 2) |
    (col("Mute_Call Rate%") > 1)
    )

volte_issue_cells.show()

+--------------+-------+----------+------+------------------+--------------------------+-----------------+-----------------------+------------------+-------------------+----+------------------+-------------------+------------------+----------+---------------+
|       Cell ID|SITE ID|      Date|Sector|Cell Availabililty|Session Setup Success Rate|VoLTE_Drop_Rate_%|Handover_Success_Rate_%|Traffic-24Hrs [GB]|DL_PRB_utilization%| CQI|IP_Throughput_Mbps|RRC Connected Users|Peak RRC Connected|Average_TA|Mute_Call Rate%|
+--------------+-------+----------+------+------------------+--------------------------+-----------------+-----------------------+------------------+-------------------+----+------------------+-------------------+------------------+----------+---------------+
| Site_A_CELL_1| Site_A|2025-01-01|   0.0|             94.81|                     99.96|              0.0|                  96.82|             95.12|              85.95|9.61|              4.32|              64.11|       

In [8]:
# --------  7. Coverage Issue Detection ---------

coverage_issue = df_clean.filter(
    (col("Average_TA") > 600) &
    (col("CQI") < 7)
    )

coverage_issue.show()

+-------+-------+----+------+------------------+--------------------------+-----------------+-----------------------+------------------+-------------------+---+------------------+-------------------+------------------+----------+---------------+
|Cell ID|SITE ID|Date|Sector|Cell Availabililty|Session Setup Success Rate|VoLTE_Drop_Rate_%|Handover_Success_Rate_%|Traffic-24Hrs [GB]|DL_PRB_utilization%|CQI|IP_Throughput_Mbps|RRC Connected Users|Peak RRC Connected|Average_TA|Mute_Call Rate%|
+-------+-------+----+------+------------------+--------------------------+-----------------+-----------------------+------------------+-------------------+---+------------------+-------------------+------------------+----------+---------------+
+-------+-------+----+------+------------------+--------------------------+-----------------+-----------------------+------------------+-------------------+---+------------------+-------------------+------------------+----------+---------------+



In [9]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

windowSpec = Window.partitionBy("Cell ID").orderBy("Date")

traffic_trend = df_clean.withColumn(
    "Prev_Traffic",
    lag("Traffic-24Hrs [GB]").over(windowSpec)
    )

traffic_trend.show()


+--------------+-------+----------+------+------------------+--------------------------+-----------------+-----------------------+------------------+-------------------+-----+------------------+-------------------+------------------+----------+---------------+------------+
|       Cell ID|SITE ID|      Date|Sector|Cell Availabililty|Session Setup Success Rate|VoLTE_Drop_Rate_%|Handover_Success_Rate_%|Traffic-24Hrs [GB]|DL_PRB_utilization%|  CQI|IP_Throughput_Mbps|RRC Connected Users|Peak RRC Connected|Average_TA|Mute_Call Rate%|Prev_Traffic|
+--------------+-------+----------+------+------------------+--------------------------+-----------------+-----------------------+------------------+-------------------+-----+------------------+-------------------+------------------+----------+---------------+------------+
| Site_A_CELL_1| Site_A|2025-01-01|   0.0|             94.81|                     99.96|              0.0|                  96.82|             95.12|              85.95| 9.61|   

In [10]:
site_daily.write.mode("overwrite").parquet(r"C:\Users\91838\Desktop\telecom_output\site_kpi_summary")

Py4JJavaError: An error occurred while calling o188.parquet.
: java.util.concurrent.ExecutionException: Boxed Exception
	at scala.concurrent.impl.Promise$.scala$concurrent$impl$Promise$$resolve(Promise.scala:99)
	at scala.concurrent.impl.Promise$DefaultPromise.tryComplete(Promise.scala:288)
	at scala.concurrent.Promise.complete(Promise.scala:57)
	at scala.concurrent.Promise.complete$(Promise.scala:56)
	at scala.concurrent.impl.Promise$DefaultPromise.complete(Promise.scala:104)
	at scala.concurrent.Promise.failure(Promise.scala:109)
	at scala.concurrent.Promise.failure$(Promise.scala:109)
	at scala.concurrent.impl.Promise$DefaultPromise.failure(Promise.scala:104)
	at org.apache.spark.sql.execution.adaptive.ResultQueryStageExec.$anonfun$doMaterialize$2(QueryStageExec.scala:336)
	at java.base/java.util.concurrent.CompletableFuture.uniWhenComplete(CompletableFuture.java:863)
	at java.base/java.util.concurrent.CompletableFuture$UniWhenComplete.tryFire(CompletableFuture.java:841)